# Lab 04-08: Apps and Interfaces

**Module:** 04 — Assembling and Deploying Applications  
**Exam Weight:** ~10 questions across Module 04 (22% of exam)  
**UI Sections:** Apps | Serving | Playground  
**Difficulty:** Beginner-Intermediate  
**Estimated Time:** 35–45 minutes

---

## Learning Objectives

- Understand the **three interfaces** for delivering GenAI applications: Playground, Serving Endpoints, and Apps
- Know when to use **Databricks Apps** (Streamlit/Gradio/Flask) for user-facing applications
- Understand how Apps connect to **serving endpoints** as the backend
- Build the architecture for a complete GenAI application: UI (App) + API (Serving) + Model
- Compare delivery options and choose the right one for each use case

---

## Exam Objectives Covered

- **Identify the appropriate interface for a given GenAI use case**
- **Describe how Databricks Apps connect to serving endpoints**
- **Distinguish between Playground, Serving Endpoints, and Apps**

---

## What Are Apps and Interfaces, and Why Do They Matter?

Building a great GenAI model is only half the battle. The other half is **getting it into the hands of users**. Databricks provides three ways to deliver your GenAI application, each suited to a different audience and use case.

| Interface | What It Is | Audience | Use Case |
|-----------|-----------|----------|----------|
| **Playground** | Interactive chat UI in Databricks workspace | Data scientists, developers | Prototyping, testing, stakeholder demos |
| **Serving Endpoints** | REST API (no UI) | Applications, other services | Backend API for apps, batch processing, integrations |
| **Databricks Apps** | Full web application (Streamlit/Gradio/Flask) | End users, business stakeholders | Production-grade user-facing applications |

The exam tests whether you understand **when to use each** and how they connect together.

---

## Mental Model

> **Analogy:** If a serving endpoint is a restaurant kitchen (takes orders, returns food), then a Databricks App is the full restaurant — kitchen plus dining room, menu, waitstaff. It is the user-facing interface that calls the kitchen (serving endpoint) behind the scenes. The Playground is like the chef's tasting counter — great for sampling and experimenting, but not where you seat 100 dinner guests.

The architecture is always the same:

```
User --> [Interface] --> [Serving Endpoint] --> [Model]

User --> Playground  --> Foundation Model API --> LLM
User --> App (Streamlit) --> Serving Endpoint --> Your Agent
Service --> REST API --> Serving Endpoint --> Your Agent
```

---

## Exam Alert: Common Traps

| # | Trap Statement | Correct Answer |
|---|---------------|----------------|
| 1 | "Serving endpoints provide a UI" | **Wrong.** Serving endpoints are REST APIs only — no UI. Use Apps or Playground for a user interface. |
| 2 | "You need external hosting for Streamlit apps" | **Wrong.** Databricks Apps can host Streamlit natively inside your workspace. No external server needed. |
| 3 | "The Playground is only for testing" | **Partially wrong.** Playground is great for prototyping AND can be shared with stakeholders for demos and quick feedback. |
| 4 | "Apps and serving endpoints are the same thing" | **Wrong.** A serving endpoint is the API backend. An App is the frontend UI that calls the endpoint. They work together. |
| 5 | "You need a separate cloud service to deploy a web app" | **Wrong.** Databricks Apps deploy directly on Databricks — no AWS/Azure/GCP web hosting needed. |

---

## Prerequisites and UI Navigation

### What You Need
- A Databricks workspace with Unity Catalog enabled
- Access to a cluster with Databricks Runtime ML
- Familiarity with serving endpoints (from Lab 04-04)

### Key UI Paths
- **UI -->** Left nav --> **Apps** --> Create and manage Databricks Apps
- **UI -->** Left nav --> **Serving** --> View and manage serving endpoints
- **UI -->** Left nav --> **Playground** --> Interactive model testing

---

## Setup

In [ ]:
# Install required packages
%pip install openai databricks-sdk -q
dbutils.library.restartPython()

In [ ]:
# ---- Imports and Configuration ----
import json
from openai import OpenAI

DATABRICKS_HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints"
)

MODEL_CHAT = "databricks-meta-llama-3-3-70b-instruct"
CATALOG = "workspace"
SCHEMA = "genai_labs"

print(f"Workspace: {DATABRICKS_HOST}")
print(f"Model:     {MODEL_CHAT}")
print("Setup complete.")

### Expected Output

```
Workspace: https://<your-workspace>.cloud.databricks.com
Model:     databricks-meta-llama-3-3-70b-instruct
Setup complete.
```

### What Just Happened

We configured the OpenAI client to point at our Databricks workspace. This is the same client that a Databricks App would use to call serving endpoints — the pattern is identical whether you call from a notebook, an App, or any other client.

---

## Step 1: The Three Interfaces — Playground, Serving Endpoints, and Apps

Let us compare the three delivery options in detail. Understanding their differences is the core exam skill for this topic.

In [ ]:
# ---- Step 1: Detailed comparison of the three interfaces ----

interfaces = [
    {
        "name": "Playground",
        "type": "Interactive chat UI",
        "audience": "Data scientists, developers, stakeholders",
        "ui": "Built-in (no code needed)",
        "access": "Databricks workspace users only",
        "customization": "Limited -- choose model, set parameters",
        "production_ready": "No -- for prototyping and demos",
        "best_for": "Quick testing, model comparison, stakeholder demos",
        "ui_path": "**UI -->** Left nav --> **Playground**"
    },
    {
        "name": "Serving Endpoints",
        "type": "REST API (no UI)",
        "audience": "Applications, services, developers",
        "ui": "None -- API only",
        "access": "Any client with API token",
        "customization": "Full -- custom models, traffic routing, scaling",
        "production_ready": "Yes -- built for production",
        "best_for": "Backend API, batch processing, service-to-service calls",
        "ui_path": "**UI -->** Left nav --> **Serving**"
    },
    {
        "name": "Databricks Apps",
        "type": "Full web application (Streamlit/Gradio/Flask)",
        "audience": "End users, business stakeholders",
        "ui": "Custom -- you build the UI",
        "access": "Shareable URL with Databricks auth",
        "customization": "Full -- custom UI, logic, branding",
        "production_ready": "Yes -- hosted on Databricks",
        "best_for": "User-facing chatbots, dashboards, internal tools",
        "ui_path": "**UI -->** Left nav --> **Apps**"
    },
]

print("=== The Three Interfaces Compared ===")
print()
for iface in interfaces:
    print(f"  {iface['name']}")
    print(f"    Type:             {iface['type']}")
    print(f"    Audience:         {iface['audience']}")
    print(f"    UI:               {iface['ui']}")
    print(f"    Access:           {iface['access']}")
    print(f"    Customization:    {iface['customization']}")
    print(f"    Production ready: {iface['production_ready']}")
    print(f"    Best for:         {iface['best_for']}")
    print(f"    Navigate to:      {iface['ui_path']}")
    print()

### Expected Output

```
=== The Three Interfaces Compared ===

  Playground
    Type:             Interactive chat UI
    Audience:         Data scientists, developers, stakeholders
    UI:               Built-in (no code needed)
    Access:           Databricks workspace users only
    ...

  Serving Endpoints
    Type:             REST API (no UI)
    ...

  Databricks Apps
    Type:             Full web application (Streamlit/Gradio/Flask)
    ...
```

### What Just Happened

We compared the three interfaces across seven dimensions. The key exam insight:
- **Playground** = prototyping (no code, workspace-only)
- **Serving Endpoints** = backend API (no UI, production-grade)
- **Apps** = full user-facing application (custom UI, calls endpoints)

---

## Step 2: Databricks Apps — Creating and Deploying a Streamlit App

Databricks Apps let you deploy **Streamlit, Gradio, Dash, or Flask** applications directly on Databricks. No external hosting required — the app runs inside your workspace and inherits Databricks authentication and security.

### UI Walkthrough: Creating a Databricks App

**UI -->** Left nav --> **Apps** --> **Create App**

1. Name your app (e.g., `customer-support-chatbot`)
2. Choose your framework: Streamlit, Gradio, Dash, or Flask
3. Upload or link your app code (from a Git repo or workspace file)
4. Configure resources (compute size, environment variables)
5. Click **Deploy** — your app gets a shareable URL

### Supported Frameworks

| Framework | Best For | Complexity |
|-----------|---------|------------|
| **Streamlit** | Quick dashboards, chatbots | Low (Python only, no HTML/CSS) |
| **Gradio** | ML model demos, input/output apps | Low (Python only) |
| **Dash** | Data-heavy dashboards with interactivity | Medium (Python + some HTML) |
| **Flask** | Full custom web apps, APIs | High (full web framework) |

In [ ]:
# ---- Step 2: Example Streamlit app code for Databricks Apps ----
# This is what you would deploy as a Databricks App.
# The app calls a serving endpoint for the LLM backend.

streamlit_app_code = '''
import streamlit as st
from openai import OpenAI
import os

# Databricks Apps automatically inject these environment variables
client = OpenAI(
    api_key=os.environ["DATABRICKS_TOKEN"],
    base_url=os.environ["DATABRICKS_HOST"] + "/serving-endpoints"
)

st.title("Customer Support Chatbot")
st.caption("Powered by Databricks Foundation Model APIs")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User input
if prompt := st.chat_input("How can I help you today?"):
    # Add user message to history
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Call the serving endpoint
    with st.chat_message("assistant"):
        response = client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                {"role": "system", "content": "You are a helpful customer support agent."},
                *st.session_state.messages
            ],
            max_tokens=500,
            stream=True  # Stream for real-time typing effect
        )
        # Stream the response
        full_response = st.write_stream(
            chunk.choices[0].delta.content or ""
            for chunk in response
            if chunk.choices[0].delta.content
        )
    st.session_state.messages.append({"role": "assistant", "content": full_response})
'''

print("=== Streamlit App Code (for Databricks Apps) ===")
print(streamlit_app_code)
print()
print("KEY POINTS:")
print("  1. Uses standard OpenAI SDK to call the serving endpoint")
print("  2. DATABRICKS_TOKEN and DATABRICKS_HOST are auto-injected")
print("  3. Streaming enabled for real-time typing effect")
print("  4. Session state maintains chat history across interactions")
print("  5. Deploy via: **UI -->** Apps --> Create App --> Upload this code")

### Expected Output

```
=== Streamlit App Code (for Databricks Apps) ===

import streamlit as st
from openai import OpenAI
import os

# Databricks Apps automatically inject these environment variables
client = OpenAI(
    api_key=os.environ["DATABRICKS_TOKEN"],
    base_url=os.environ["DATABRICKS_HOST"] + "/serving-endpoints"
)
...

KEY POINTS:
  1. Uses standard OpenAI SDK to call the serving endpoint
  2. DATABRICKS_TOKEN and DATABRICKS_HOST are auto-injected
  3. Streaming enabled for real-time typing effect
  4. Session state maintains chat history across interactions
  5. Deploy via: **UI -->** Apps --> Create App --> Upload this code
```

### What Just Happened

We wrote a complete **Streamlit chatbot application** that could be deployed as a Databricks App. The architecture is clean: the Streamlit app (frontend) calls the serving endpoint (backend) using the OpenAI SDK. Databricks automatically provides authentication tokens — no manual secret management needed.

---

## Step 3: Connecting an App to a Serving Endpoint — The Architecture

Every Databricks App follows the same architecture pattern. Understanding this architecture is key for the exam.

```
+-------------------+          +--------------------+          +------------------+
|    End User        |          |  Databricks App    |          | Serving Endpoint |
|   (browser)        |  HTTPS   | (Streamlit/Gradio) |  REST    | (your model/     |
|                    | -------> |                    | -------> |  agent/chain)    |
|                    | <------- |                    | <------- |                  |
+-------------------+          +--------------------+          +------------------+
       ^                              ^                              ^
       |                              |                              |
   Databricks                   Runs on                      Foundation Model API
   auth (SSO)                 Databricks                    or Custom Endpoint
```

In [ ]:
# ---- Step 3: Demonstrate the App --> Endpoint connection ----
# This simulates what a Databricks App does behind the scenes.

# Step 1: The App receives a user message
user_message = "I need to return a product I bought last week. What's the process?"

# Step 2: The App calls the serving endpoint (same as what Streamlit does)
response = client.chat.completions.create(
    model=MODEL_CHAT,
    messages=[
        {"role": "system", "content": "You are a helpful customer support agent. Be concise and friendly."},
        {"role": "user", "content": user_message}
    ],
    max_tokens=300,
    temperature=0.3
)

# Step 3: The App displays the response to the user
assistant_response = response.choices[0].message.content

print("=== App --> Serving Endpoint Flow ===")
print()
print("Step 1: User types in the App UI:")
print(f"  '{user_message}'")
print()
print(f"Step 2: App calls serving endpoint ({MODEL_CHAT}):")
print(f"  POST {DATABRICKS_HOST}/serving-endpoints/{MODEL_CHAT}/invocations")
print(f"  Tokens used: {response.usage.total_tokens}")
print()
print("Step 3: App displays response to user:")
print(f"  '{assistant_response[:200]}...'")
print()
print("This is the EXACT flow that happens in every Databricks App.")

### Expected Output

```
=== App --> Serving Endpoint Flow ===

Step 1: User types in the App UI:
  'I need to return a product I bought last week. What's the process?'

Step 2: App calls serving endpoint (databricks-meta-llama-3-3-70b-instruct):
  POST https://<workspace>/serving-endpoints/databricks-meta-llama-3-3-70b-instruct/invocations
  Tokens used: 185

Step 3: App displays response to user:
  'I'd be happy to help you with your return! Here's the process...

This is the EXACT flow that happens in every Databricks App.
```

### What Just Happened

We demonstrated the **three-step flow** that every Databricks App follows:
1. User interacts with the App UI (Streamlit, Gradio, etc.)
2. App calls the serving endpoint via REST API (using OpenAI SDK)
3. App displays the endpoint's response to the user

This separation of concerns (UI vs API) is a fundamental architecture pattern.

---

## Step 4: Decision Matrix — App vs Playground vs Serving Endpoint

This is the most exam-tested concept in this lab. Given a scenario, choose the right interface.

In [ ]:
# ---- Step 4: Exam-style practice -- choose the right interface ----

scenarios = [
    {
        "scenario": "A data scientist wants to quickly compare Llama 3.3 vs Mixtral responses.",
        "answer": "Playground",
        "why": "Quick model comparison with zero code -- perfect for Playground"
    },
    {
        "scenario": "A customer support team needs a chatbot with a friendly UI.",
        "answer": "Databricks App (Streamlit)",
        "why": "End users need a polished UI -- Streamlit App calling a serving endpoint"
    },
    {
        "scenario": "A microservice needs to call an LLM to classify incoming emails.",
        "answer": "Serving Endpoint (REST API)",
        "why": "Service-to-service call -- no UI needed, just the REST API"
    },
    {
        "scenario": "A product manager wants to demo the RAG agent to leadership.",
        "answer": "Playground (or a simple App)",
        "why": "Playground for a quick demo; App if they want a branded experience"
    },
    {
        "scenario": "An e-commerce platform wants to integrate AI recommendations into its website.",
        "answer": "Serving Endpoint (REST API)",
        "why": "External website calls the endpoint -- no Databricks UI needed"
    },
    {
        "scenario": "An internal HR team needs a self-service FAQ bot for employees.",
        "answer": "Databricks App (Streamlit/Gradio)",
        "why": "Internal users need a friendly web UI hosted on Databricks"
    },
    {
        "scenario": "A batch pipeline processes 10,000 documents through an LLM nightly.",
        "answer": "Serving Endpoint (REST API)",
        "why": "Batch processing -- call the API programmatically, no UI needed"
    },
]

print("=== Exam Practice: Choose the Right Interface ===")
print()
for i, s in enumerate(scenarios, 1):
    print(f"Scenario {i}: {s['scenario']}")
    print(f"  --> {s['answer']}")
    print(f"      {s['why']}")
    print()

### Expected Output

```
=== Exam Practice: Choose the Right Interface ===

Scenario 1: A data scientist wants to quickly compare Llama 3.3 vs Mixtral responses.
  --> Playground
      Quick model comparison with zero code -- perfect for Playground

Scenario 2: A customer support team needs a chatbot with a friendly UI.
  --> Databricks App (Streamlit)
      End users need a polished UI -- Streamlit App calling a serving endpoint

Scenario 3: A microservice needs to call an LLM to classify incoming emails.
  --> Serving Endpoint (REST API)
      Service-to-service call -- no UI needed, just the REST API

...(remaining scenarios)...
```

### What Just Happened

We practiced the **exam-critical skill** of choosing the right interface for a given scenario. The decision rule is:
- **Need to test/prototype quickly?** --> Playground
- **Need a user-facing UI?** --> Databricks App
- **Need a programmable API?** --> Serving Endpoint

---

## Step 5: Sharing and Access Control for Apps

Databricks Apps inherit the workspace's security model. Access control determines who can see and use your app.

In [ ]:
# ---- Step 5: Access control for Databricks Apps ----

access_model = {
    "authentication": [
        {"method": "Databricks SSO", "description": "Users log in with their Databricks workspace credentials"},
        {"method": "OAuth/SAML", "description": "Enterprise SSO via identity provider (Okta, Azure AD, etc.)"},
    ],
    "permissions": [
        {"level": "CAN VIEW", "grants": "User can access the app and interact with it"},
        {"level": "CAN MANAGE", "grants": "User can edit, redeploy, and configure the app"},
        {"level": "IS OWNER", "grants": "Full control including deletion and permission management"},
    ],
    "sharing_options": [
        {"option": "Workspace URL", "description": "Share the app URL with workspace users"},
        {"option": "Group permissions", "description": "Grant access to entire groups (e.g., 'support-team')"},
        {"option": "Service principal", "description": "Allow automated systems to access the app"},
    ]
}

print("=== Databricks Apps: Access Control ===")
print()
print("Authentication:")
for auth in access_model['authentication']:
    print(f"  - {auth['method']}: {auth['description']}")

print()
print("Permission Levels:")
for perm in access_model['permissions']:
    print(f"  - {perm['level']}: {perm['grants']}")

print()
print("Sharing Options:")
for share in access_model['sharing_options']:
    print(f"  - {share['option']}: {share['description']}")

print()
print("KEY EXAM POINT: Apps inherit Databricks security.")
print("No separate auth system needed -- workspace SSO handles it.")

### Expected Output

```
=== Databricks Apps: Access Control ===

Authentication:
  - Databricks SSO: Users log in with their Databricks workspace credentials
  - OAuth/SAML: Enterprise SSO via identity provider (Okta, Azure AD, etc.)

Permission Levels:
  - CAN VIEW: User can access the app and interact with it
  - CAN MANAGE: User can edit, redeploy, and configure the app
  - IS OWNER: Full control including deletion and permission management

Sharing Options:
  - Workspace URL: Share the app URL with workspace users
  - Group permissions: Grant access to entire groups (e.g., 'support-team')
  - Service principal: Allow automated systems to access the app

KEY EXAM POINT: Apps inherit Databricks security.
No separate auth system needed -- workspace SSO handles it.
```

### What Just Happened

We reviewed the **access control model** for Databricks Apps. The key exam insight: Apps use the same authentication and authorization as the rest of Databricks. No external identity provider configuration needed (unless your workspace already uses one).

---

## Complete Architecture: From Model to User

Let us put everything together — the full architecture of a production GenAI application on Databricks.

```
+---------------------------------------------------------------------+
|                    DATABRICKS WORKSPACE                              |
|                                                                     |
|  +-------------------+     +---------------------+                  |
|  | Databricks App    |     | Serving Endpoint    |                  |
|  | (Streamlit UI)    | --> | (Model/Agent/Chain)  |                 |
|  |                   |     |                     |                  |
|  | - Chat interface  |     | - Foundation Model  |                  |
|  | - Session state   |     | - Custom PyFunc     |                  |
|  | - User auth       |     | - LangChain agent   |                  |
|  +-------------------+     +---------------------+                  |
|          ^                          |                               |
|          |                          v                               |
|     End Users               +-------------------+                   |
|    (browser)                | Unity Catalog     |                   |
|                             | - Model registry  |                   |
|                             | - Vector Search   |                   |
|                             | - MCP tools       |                   |
|                             +-------------------+                   |
+---------------------------------------------------------------------+
```

---

## Exam Tips and Traps

| # | Tip or Trap | Explanation |
|---|-------------|-------------|
| 1 | **Trap:** "Serving endpoints have a built-in UI" | Serving endpoints are REST APIs only. Use Playground or Apps for UI. |
| 2 | **Tip:** Databricks Apps host Streamlit/Gradio/Flask natively | No external hosting (Heroku, AWS, etc.) needed. |
| 3 | **Trap:** "You need separate authentication for Apps" | Apps inherit Databricks workspace SSO. Same login, same permissions. |
| 4 | **Tip:** The architecture is always App --> Endpoint --> Model | The App is the frontend, the endpoint is the backend. Separation of concerns. |
| 5 | **Trap:** "Playground cannot be shared with stakeholders" | Playground sessions can be shared within the workspace for demos and feedback. |
| 6 | **Tip:** For the exam, "need a UI for end users" = Databricks App | If the question says "non-technical users" or "customer-facing," the answer is App. |

---

## Key Takeaways

### Core Concepts

- **Playground**: Built-in chat UI for prototyping and demos — no code required
- **Serving Endpoints**: REST API for production — the backend that Apps and services call
- **Databricks Apps**: Full web applications (Streamlit, Gradio, Flask) hosted on Databricks — the user-facing frontend
- Apps call serving endpoints via the OpenAI SDK — same code pattern everywhere
- Apps inherit Databricks authentication — no separate auth system needed

### Decision Rule

```
Quick test/prototype?          --> Playground
Need a programmable API?       --> Serving Endpoint
Need a user-facing UI?         --> Databricks App
Service-to-service call?       --> Serving Endpoint
Non-technical stakeholders?    --> Databricks App (or Playground for demos)
```

### Architecture Pattern

```
User --> Databricks App (UI) --> Serving Endpoint (API) --> Model/Agent
```

---

## Cleanup

This lab did not create any persistent resources. The only resources used were pay-per-token API calls, which have no ongoing cost.

---

## Next Lab

**Lab 04-09: Code Your Own Agent** -- you'll learn how to build a custom agent
using open-source libraries (OpenAI Agents SDK, LangGraph, DSPy) and deploy it
as a production application on Databricks Apps with a built-in chat UI.

> Continue to `04_assembling_deploying/09_code_your_own_agent.ipynb`